# 03 · Build a labelled training table

Combines vibration features with geometry labels: each accepted wheel-group
feature row gets a `y` = quality index interpolated at the recording's day
for its sensor location. This is the table the model will train on.

_Note: extraction is ~2 s/log; raise the slice or parallelise for the full set._


In [ ]:
import sys, os
# point Python at the package source
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np, pandas as pd
from intelliswitch import Config

DATA_ROOT = r"/Volumes/Seagate Expansion Drive/Mohammad/M Drive DTU/MATLAB Scripts/IntelliswitchScripts"
W10  = os.path.join(DATA_ROOT, "W10v140")        # vibration .log
BLADE = os.path.join(DATA_ROOT, "Bladenr8252")   # geometry .csv
cfg = Config()
cfg.freq_min_hz, cfg.freq_max_hz, cfg.geom_method, cfg.geom_sensor_locations_m


In [ ]:
from pathlib import Path
from intelliswitch.features import extract_features
from intelliswitch.geometry import align_campaigns, build_deg_model
from intelliswitch.geometry.deg_model import attach_labels

deg = build_deg_model(align_campaigns(BLADE, cfg), cfg)

frames = []
for p in sorted(Path(W10).glob('*.log'))[:40]:
    r = extract_features(p, cfg)
    if r.accepted:
        frames.append(r.features)
data = pd.concat(frames, ignore_index=True)
data = attach_labels(data, deg, cfg)
print('rows:', len(data), '| finite labels:', int(np.isfinite(data['y']).sum()))
data[['source','sensor','days','speed_kmh','y']].head(20)


### Save for modelling


In [ ]:
# data.to_parquet('training_table.parquet')  # uncomment to persist
data['y'].describe()
